<a href="https://colab.research.google.com/github/yashikasgh/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashikasgh/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

##1. My Rule

A content page should be reviewed for refresh if it has high search visibility, a low CTR, and has not been updated for a long time. These pages have the highest potential to improve because many users are seeing them, but relatively few are clicking on them.

### Reason Codes

- **STALE_CONTENT** – The page has not been updated for a long time.
- **HIGH_VISIBILITY** – The page receives a high number of search impressions.
- **LOW_CTR** – The page has a lower-than-average click-through rate.

In [1]:
import pandas as pd
import os

# Load the starter dataset
df = pd.read_csv("content_refresh_anonymized.csv")

# Rule conditions
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["high_visibility"] = (df["impressions_90d"] >= 500).astype(int)
df["low_ctr"] = (df["ctr"] < df["ctr"].median()).astype(int)

# Baseline score
df["baseline_score"] = (
    df["stale"] * 40 +
    df["high_visibility"] * 40 +
    df["low_ctr"] * 20
)

# Reason codes
def get_reason(row):
    reasons = []

    if row["stale"]:
        reasons.append("STALE_CONTENT")
    if row["high_visibility"]:
        reasons.append("HIGH_VISIBILITY")
    if row["low_ctr"]:
        reasons.append("LOW_CTR")

    return ", ".join(reasons) if reasons else "NONE"

df["reason_code"] = df.apply(get_reason, axis=1)

# Action labels
def get_action(score):
    if score >= 80:
        return "Refresh Now"
    elif score >= 40:
        return "Review"
    else:
        return "No Action"

df["action"] = df["baseline_score"].apply(get_action)

# Rank pages
baseline_queue = df.sort_values(
    "baseline_score",
    ascending=False
)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

baseline_queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

baseline_queue.head(10)

CSV saved successfully!


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,impression_tier,position_tier,trend_direction,trend_pct,stale,high_visibility,low_ctr,baseline_score,reason_code,action
698,content_b16bd7307b39,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4329.0,27844.0,...,good,page_3_5,down,-69.7,1,1,1,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",Refresh Now
3507,content_074ba6ead17b,client_d029fa3a95,0.0,0.0,LOW,0.0,keyword article,informational,3994.0,27901.0,...,moderate,page_3_5,down,-36.5,1,1,1,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",Refresh Now
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,good,page_3_5,down,-89.1,1,1,1,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",Refresh Now
21268,content_0a91db491d14,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3478.0,21948.0,...,good,striking,down,-51.8,1,1,0,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
23215,content_bdbec75c1148,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3696.0,24643.0,...,moderate,page_3_5,stable,-11.7,1,1,0,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
5327,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3388.0,21742.0,...,good,striking,down,-52.2,1,1,0,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
26799,content_77d4d5930e5e,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4020.0,26513.0,...,moderate,striking,down,-55.0,1,1,0,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
7021,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3861.0,24672.0,...,good,page_3_5,down,-74.7,1,1,0,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
12045,content_c2d929d83eaa,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4758.0,30070.0,...,good,striking,down,-62.8,1,1,0,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
22872,content_e3ff1b093148,client_d029fa3a95,0.0,0.0,LOW,0.0,keyword article,informational,4758.0,33575.0,...,moderate,page_1,down,-68.5,1,1,0,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now


## 2. Build the ranked queue (writes the CSV)

I implemented a simple rule-based baseline to rank content pages for refresh. Each page receives a baseline score based on three signals: content freshness, search visibility, and CTR. The ranked queue also includes a reason code and an action label, and it is saved as `work/outputs/baseline_action_score.csv`.

In [3]:
import pandas as pd
import os

# Load starter dataset
df = pd.read_csv("content_refresh_anonymized.csv")

# Rule conditions
impression_threshold = df["impressions_90d"].median()

df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["high_visibility"] = (df["impressions_90d"] >= impression_threshold).astype(int)
df["low_ctr"] = (df["ctr"] < df["ctr"].median()).astype(int)

# Baseline score
df["baseline_score"] = (
    40 * df["stale"] +
    40 * df["high_visibility"] +
    20 * df["low_ctr"]
)

# Reason codes
def get_reason(row):
    reasons = []

    if row["stale"]:
        reasons.append("STALE_CONTENT")
    if row["high_visibility"]:
        reasons.append("HIGH_VISIBILITY")
    if row["low_ctr"]:
        reasons.append("LOW_CTR")

    return ", ".join(reasons) if reasons else "NONE"

df["reason_code"] = df.apply(get_reason, axis=1)

# Action labels
def get_action(score):
    if score >= 80:
        return "Refresh Now"
    elif score >= 40:
        return "Review"
    else:
        return "No Action"

df["action"] = df["baseline_score"].apply(get_action)

# Ranked queue
baseline_queue = df.sort_values(
    by="baseline_score",
    ascending=False
)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

baseline_queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved to work/outputs/baseline_action_score.csv")

baseline_queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(20)

CSV saved to work/outputs/baseline_action_score.csv


,content_id,baseline_score,reason_code,action
11489,content_5feee3994adb,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",Refresh Now
698,content_b16bd7307b39,100,"STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR",Refresh Now
5327,content_fe16a55cd13d,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
26810,content_ecb6215e79fd,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
16514,content_7368877ea310,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
26840,content_7f116ae1f6f5,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
16751,content_cf56e2e2e282,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
23215,content_bdbec75c1148,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
12045,content_c2d929d83eaa,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now
26799,content_77d4d5930e5e,80,"STALE_CONTENT, HIGH_VISIBILITY",Refresh Now


## 3. Top-20 review

I reviewed the top 20 pages recommended by the baseline rule.

- **Action:** Pages with the highest scores are marked as **Refresh Now**, while medium-scoring pages are marked as **Review**.
- **Reason Codes:** The recommendations are based on one or more of these signals: `STALE_CONTENT`, `HIGH_VISIBILITY`, and `LOW_CTR`.
- **Confidence:** I have medium confidence in these recommendations because they are based on transparent rules using historical data.
- **What would make them wrong?** Some pages may have already been updated recently outside this dataset, may be seasonal, or may have low CTR due to search intent rather than poor content quality. Therefore, these recommendations should be reviewed by an editor before taking action.

In [5]:
for _, row in top20.iterrows():
    print(
        f"Content: {row['content_id']} | "
        f"Action: {row['action']} | "
        f"Reason: {row['reason_code']} | "
        f"Confidence: Medium | "
        f"What would make it wrong: The page may already have been updated recently, be seasonal, or have low CTR due to user search intent."
    )

Content: content_5feee3994adb | Action: Refresh Now | Reason: STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR | Confidence: Medium | What would make it wrong: The page may already have been updated recently, be seasonal, or have low CTR due to user search intent.
Content: content_b16bd7307b39 | Action: Refresh Now | Reason: STALE_CONTENT, HIGH_VISIBILITY, LOW_CTR | Confidence: Medium | What would make it wrong: The page may already have been updated recently, be seasonal, or have low CTR due to user search intent.
Content: content_fe16a55cd13d | Action: Refresh Now | Reason: STALE_CONTENT, HIGH_VISIBILITY | Confidence: Medium | What would make it wrong: The page may already have been updated recently, be seasonal, or have low CTR due to user search intent.
Content: content_ecb6215e79fd | Action: Refresh Now | Reason: STALE_CONTENT, HIGH_VISIBILITY | Confidence: Medium | What would make it wrong: The page may already have been updated recently, be seasonal, or have low CTR due to user search in

##4. Weak Picks

Some pages selected by the baseline rule may not actually need a content refresh. For example:

- Pages with high impressions but naturally low CTR because of the search intent.
- Seasonal or event-based content that is expected to lose traffic after a certain period.
- Pages that have already been updated recently, but the dataset does not yet reflect those changes.

These pages could receive a high baseline score even though refreshing them may not improve performance.

##5. Leakage Check

I confirmed that my baseline rule only uses information available before making the refresh decision.

Features used:
- days_since_last_update
- impressions_90d
- ctr

I did **not** use:
- trend_direction
- trend_pct
- is_declining_label (derived label)
- Any future performance information or product flags.

Therefore, the baseline does not contain label leakage or future-window information.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.